# mlflow model registry, deeper pass

the 2021 walkthrough was just `log_model`. now i actually want to use the registry: stage transitions (None -> Staging -> Production), aliasing, loading by stage.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)
clf = LogisticRegression(max_iter=500).fit(X, y)

mlflow.set_experiment('iris-registry-demo')
with mlflow.start_run() as run:
    mlflow.log_param('max_iter', 500)
    mlflow.log_metric('train_acc', clf.score(X, y))
    mlflow.sklearn.log_model(clf, 'model', registered_model_name='iris-lr')
    rid = run.info.run_id
print(rid)


In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

# move latest version to staging
latest = client.get_latest_versions('iris-lr', stages=['None'])[0]
client.transition_model_version_stage('iris-lr', latest.version, 'Staging')


In [ ]:
# load by stage at inference time
import mlflow.pyfunc
model = mlflow.pyfunc.load_model('models:/iris-lr/Staging')
print(model.predict(X[:3]))


the `models:/<name>/<stage>` uri is the actual valuable bit. our serving image can pin to a stage and never know about run ids.

trying lr=3e-4 vs 1e-3, the higher one diverges on this dataset.

In [ ]:
# log to wandb
# import wandb
# wandb.init(project='ml-experiments')


smaller batch helped on this one. counterintuitive.

found a corner case in the input pipeline. fixed.